In [14]:
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.decomposition import PCA

In [15]:
delimiter = "|"

theta_df = pd.read_csv("../data/output/LDA_THETA_ENRICHED.csv")
lib_df = pd.read_csv("../data/output/LIB.csv", sep=delimiter)

theta_df.head()

,T00,T01,T02,T03,T04,Document,BookNum,Book,Dominant_Topic
0,0.067189,0.73281,0.066667,0.066667,0.066667,10_10_1,10,2 Samuel,T01
1,0.200000,0.20000,0.200000,0.200000,0.200000,10_10_10,10,2 Samuel,T00
2,0.200000,0.20000,0.200000,0.200000,0.200000,10_10_11,10,2 Samuel,T00
3,0.399998,0.40000,0.066667,0.066667,0.066669,10_10_12,10,2 Samuel,T01
4,0.100000,0.10000,0.100000,0.100001,0.599999,10_10_13,10,2 Samuel,T04


In [16]:
topic_cols = [col for col in theta_df.columns if col.startswith("T")]

theta_matrix = theta_df[topic_cols].values

pca = PCA(n_components=2)
theta_pca = pca.fit_transform(theta_matrix)

theta_pca_df = pd.DataFrame(
    theta_pca,
    columns=["PC1", "PC2"]
)

theta_pca_df["BookNum"] = theta_df["BookNum"].astype(int)
theta_pca_df["Document"] = theta_df["Document"]
theta_pca_df["Dominant_Topic"] = theta_df["Dominant_Topic"]

theta_pca_df.head()

,PC1,PC2,BookNum,Document,Dominant_Topic
0,0.505477,0.236018,10,10_10_1,T01
1,-0.013437,-0.053370,10,10_10_10,T00
2,-0.013437,-0.053370,10,10_10_11,T00
3,0.040227,0.306022,10,10_10_12,T01
4,-0.027697,-0.223791,10,10_10_13,T04


In [17]:
topic_means = theta_df[topic_cols].mean()

mean_weight_map = topic_means.to_dict()

theta_pca_df["Point_Size"] = (
    theta_pca_df["Dominant_Topic"]
    .map(mean_weight_map)
    * 2000
)

In [18]:
lib_df["book_number"] = lib_df["book_number"].astype(int)

theta_pca_df = theta_pca_df.merge(
    lib_df[["book_number", "book_name", "testament"]],
    left_on="BookNum",
    right_on="book_number",
    how="left"
)

theta_pca_df.rename(columns={
    "book_name": "BookName",
    "testament": "Testament"
}, inplace=True)

theta_pca_df.head()

,PC1,PC2,BookNum,Document,Dominant_Topic,Point_Size,book_number,BookName,Testament
0,0.505477,0.236018,10,10_10_1,T01,469.313356,10,2 Samuel,Old Testament
1,-0.013437,-0.053370,10,10_10_10,T00,453.547551,10,2 Samuel,Old Testament
2,-0.013437,-0.053370,10,10_10_11,T00,453.547551,10,2 Samuel,Old Testament
3,0.040227,0.306022,10,10_10_12,T01,469.313356,10,2 Samuel,Old Testament
4,-0.027697,-0.223791,10,10_10_13,T04,351.583734,10,2 Samuel,Old Testament


In [19]:
theta_pca_df["Testament"].value_counts()

Testament
Old Testament    23145
New Testament     7957
Name: count, dtype: int64

In [21]:
theta_pca_df.head()

,PC1,PC2,BookNum,Document,Dominant_Topic,Point_Size,book_number,BookName,Testament
0,0.505477,0.236018,10,10_10_1,T01,469.313356,10,2 Samuel,Old Testament
1,-0.013437,-0.053370,10,10_10_10,T00,453.547551,10,2 Samuel,Old Testament
2,-0.013437,-0.053370,10,10_10_11,T00,453.547551,10,2 Samuel,Old Testament
3,0.040227,0.306022,10,10_10_12,T01,469.313356,10,2 Samuel,Old Testament
4,-0.027697,-0.223791,10,10_10_13,T04,351.583734,10,2 Samuel,Old Testament


In [22]:
fig_theta = go.Figure()

books_sorted = (
    theta_pca_df
    .sort_values("BookNum")
    ["BookName"]
    .unique()
)

for book in books_sorted:

    subset = theta_pca_df[theta_pca_df["BookName"] == book]

    subset_ot = subset[subset["Testament"] == "Old Testament"]
    subset_nt = subset[subset["Testament"] == "New Testament"]

    if not subset_ot.empty:
        fig_theta.add_trace(
            go.Scatter(
                x=subset_ot["PC1"],
                y=subset_ot["PC2"],
                mode="markers",
                name=book,
                marker=dict(
                    size=subset_ot["Point_Size"] / 40,
                    opacity=0.7
                ),
                text=subset_ot["Document"],
                hovertemplate=(
                    "Document: %{text}<br>"
                    "Book: " + book + "<br>"
                    "Testament: Old Testament<br>"
                    "PC1: %{x:.3f}<br>"
                    "PC2: %{y:.3f}<extra></extra>"
                )
            )
        )

    if not subset_nt.empty:
        fig_theta.add_trace(
            go.Scatter(
                x=subset_nt["PC1"],
                y=subset_nt["PC2"],
                mode="markers",
                name=book if subset_ot.empty else None,
                marker=dict(
                    size=subset_nt["Point_Size"] / 40,
                    opacity=0.7,
                    symbol="circle-open"
                ),
                text=subset_nt["Document"],
                hovertemplate=(
                    "Document: %{text}<br>"
                    "Book: " + book + "<br>"
                    "Testament: New Testament<br>"
                    "PC1: %{x:.3f}<br>"
                    "PC2: %{y:.3f}<extra></extra>"
                ),
                showlegend=subset_ot.empty
            )
        )

fig_theta.update_layout(
    title="LDA Theta PCA Visualization<br>Filled = Old Testament, Hollow = New Testament",
    width=1400,
    height=900,
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.25,
        xanchor="center",
        x=0.5,
        font=dict(size=9)
    ),
    xaxis_title="PC1",
    yaxis_title="PC2"
)

fig_theta

In [23]:
save_path = "/Users/nicholasthornton/Downloads/DS 5001/DS-5001-Bible-Analysis-Final-Project/graphs"

os.makedirs(save_path, exist_ok=True)

fig_theta.write_html(
    os.path.join(save_path, "LDA_theta_PCA_PC1_PC2_by_book_and_testament.html")
)

print("LDA Theta PCA plot saved.")

LDA Theta PCA plot saved.


In [24]:
pca.explained_variance_ratio_[:2]

array([0.32382437, 0.26171653])

In [25]:
theta_pca_df.groupby("Testament")[["PC1", "PC2"]].mean()

,PC1,PC2
Testament,,
New Testament,-0.004003,0.007219
Old Testament,0.001376,-0.002482


In [26]:
theta_pca_df.groupby("Testament")[["PC1", "PC2"]].std()

,PC1,PC2
Testament,,
New Testament,0.257042,0.232267
Old Testament,0.251419,0.225568


In [29]:
len(pca.components_[0])

5

In [30]:
pd.Series(
    pca.components_[0],
    index=topic_cols
).sort_values()

pd.Series(
    pca.components_[1],
    index=topic_cols
).sort_values()

T02   -0.434353
T04   -0.340842
T03   -0.302988
T01    0.433919
T00    0.644265
dtype: float64

The first two principal components explain approximately 58.5% of the variance in document-level topic mixtures, indicating that the projection captures a substantial portion of corpus structure. The triangular geometry suggests that documents are constrained by tradeoffs among three dominant thematic directions. PC1 represents a strong contrast between Topics T00 and T01 versus Topics T02, T03, and T04, indicating that documents positioned along the right side of the triangle are dominated by the first pair of topics, while documents on the left emphasize the latter group. This confirms that the primary axis of variation reflects a structured thematic opposition rather than random dispersion. PC2 captures an additional independent contrast, contributing to the triangular simplex structure observed in the projection. Importantly, Old and New Testament documents have nearly identical centroids and comparable dispersion along both components, indicating that the structure is not driven by simple Testament-level separation. Instead, both Testaments participate in the same underlying thematic tradeoff space. Documents near the vertices exhibit strong dominance by specific topics, while documents near the center reflect more balanced mixtures. Overall, the PCA projection reveals that the corpus is organized around competing latent thematic poles rather than discrete clusters.